# 02 — FMA Data Ingestion & Preprocessing

**Goal:** Produce reproducible, leakage-safe audio feature arrays for the raw-feature baseline and autoencoder. No PCA is used.

**Input:** `data/fma_metadata/features.csv` and `tracks.csv` from [FMA](https://github.com/mdeff/fma).  
**Output:** `data/processed/training.npz`, `validation.npz`, `test.npz`, plus metadata and preprocessing artifacts.

Run cells in order. The notebook works from VS Code/Jupyter or Colab after cloning the repository and downloading/extracting `fma_metadata.zip`.


## 1. Imports and project paths

In [ ]:
# Run once if your environment is missing dependencies:
# %pip install numpy pandas scikit-learn matplotlib joblib

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Find the repository root whether the notebook runs from the root or notebooks/.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find repository root (folder containing data/). "
        "In Colab, clone the repository and %cd into it first."
    )
METADATA_DIR = PROJECT_ROOT / "data" / "fma_metadata"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_FILE = METADATA_DIR / "features.csv"
TRACKS_FILE = METADATA_DIR / "tracks.csv"
for path in (FEATURES_FILE, TRACKS_FILE):
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Extract fma_metadata.zip into {METADATA_DIR}")
print("Project root:", PROJECT_ROOT)
print("Metadata:", METADATA_DIR)
print("Output:", OUTPUT_DIR)

## 2. Load FMA CSV files

FMA uses multi-level column headers: three header rows for `features.csv` and two for `tracks.csv`. Track IDs are the row index.

In [ ]:
features = pd.read_csv(FEATURES_FILE, index_col=0, header=[0, 1, 2])
tracks = pd.read_csv(TRACKS_FILE, index_col=0, header=[0, 1])
features.index = pd.to_numeric(features.index, errors="raise").astype("int64")
tracks.index = pd.to_numeric(tracks.index, errors="raise").astype("int64")
assert features.index.is_unique and tracks.index.is_unique, "Duplicate track IDs"
print("All features:", features.shape)
print("All tracks:", tracks.shape)
display(features.iloc[:3, :6])
display(tracks.iloc[:3, :6])

## 3. Select FMA-small and align track IDs

The FMA metadata uses `track/subset` to identify the small subset and `set/split` for its official split. This notebook uses those provided splits.

In [ ]:
required = [("set", "subset"), ("set", "split")]
for col in required:
    if col not in tracks.columns:
        raise KeyError(f"Expected FMA column {col!r} not found")
subset_mask = tracks[("set", "subset")].astype("string").str.lower().eq("small")
subset_tracks = tracks.loc[subset_mask].copy()
common_ids = subset_tracks.index.intersection(features.index, sort=False)
subset_tracks = subset_tracks.loc[common_ids].copy()
X_raw = features.loc[common_ids].copy()
print("FMA-small tracks in metadata:", int(subset_mask.sum()))
print("Aligned tracks:", len(common_ids))
print("Feature count:", X_raw.shape[1])
print("Splits:")
print(subset_tracks[("set", "split")].value_counts(dropna=False))
if len(common_ids) == 0:
    raise ValueError("No matching FMA-small tracks found")

## 4. Validate feature matrix and splits

Convert all feature values to numeric. Missing and infinite values will be handled **after** the split to avoid learning preprocessing statistics from validation or test data.

In [ ]:
X_raw = X_raw.apply(pd.to_numeric, errors="coerce")
X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
missing = X_raw.isna().sum().sum()
print("Missing/invalid feature cells:", int(missing))
print("Tracks with any missing feature:", int(X_raw.isna().any(axis=1).sum()))

split_values = subset_tracks[("set", "split")].astype("string").str.lower().str.strip()
expected = {"training", "validation", "test"}
unknown = set(split_values.dropna().unique()) - expected
if unknown or split_values.isna().any():
    raise ValueError(f"Unexpected split values: {unknown}; missing: {split_values.isna().sum()}")
for split in ("training", "validation", "test"):
    print(split, int((split_values == split).sum()))
    if not (split_values == split).any():
        raise ValueError(f"No samples in {split} split")

## 5. Split before fitting preprocessing

Drop feature columns that are entirely missing in the **training** split, and constant columns based only on training data. Use median imputation and standard scaling fitted on training only.

In [ ]:
split_masks = {s: split_values.eq(s).to_numpy(dtype=bool) for s in ("training", "validation", "test")}
train_df = X_raw.iloc[np.flatnonzero(split_masks["training"])]
valid_columns = train_df.notna().any(axis=0)
train_nonempty = train_df.loc[:, valid_columns]
# Training columns with <=1 distinct non-null value do not provide variation.
varying_columns = train_nonempty.nunique(dropna=True).gt(1)
keep_columns = train_nonempty.columns[varying_columns]
if len(keep_columns) == 0:
    raise ValueError("No usable audio feature columns remain")
X_selected = X_raw.loc[:, keep_columns]
print("Original features:", X_raw.shape[1])
print("Retained features:", X_selected.shape[1])
print("Dropped features:", X_raw.shape[1] - X_selected.shape[1])

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
train_array = imputer.fit_transform(X_selected.loc[split_masks["training"]])
scaler.fit(train_array)

processed = {}
for split in ("training", "validation", "test"):
    ids = X_selected.index[split_masks[split]].to_numpy(dtype=np.int64)
    raw = X_selected.loc[ids]
    transformed = scaler.transform(imputer.transform(raw)).astype(np.float32)
    if not np.isfinite(transformed).all():
        raise ValueError(f"Nonfinite processed features in {split}")
    processed[split] = {"X": transformed, "track_ids": ids}
    print(split, transformed.shape, transformed.dtype)

## 6. Save arrays, metadata, and preprocessing artifacts

The `.npz` contract for your teammate is `X` (float32 matrix) and `track_ids` (int64 vector). All three splits have the same feature order.

In [ ]:
for split, arrays in processed.items():
    destination = OUTPUT_DIR / f"{split}.npz"
    np.savez_compressed(destination, **arrays)
    print("Saved:", destination)

# Store metadata as a flat-column CSV, indexed by track ID, for later evaluation.
metadata = subset_tracks.copy()
metadata.columns = ["__".join(map(str, col)) for col in metadata.columns]
metadata.index.name = "track_id"
metadata.to_csv(OUTPUT_DIR / "track_metadata.csv")

# Store original FMA feature labels and order as JSON lists.
feature_labels = [list(map(str, col)) for col in keep_columns.to_list()]
(OUTPUT_DIR / "feature_columns.json").write_text(json.dumps(feature_labels, indent=2), encoding="utf-8")
joblib.dump({"imputer": imputer, "scaler": scaler, "feature_columns": keep_columns},
            OUTPUT_DIR / "preprocessing.joblib")
print("Saved metadata and preprocessing artifacts")

## 7. Sanity checks

In [ ]:
loaded = {s: np.load(OUTPUT_DIR / f"{s}.npz") for s in processed}
assert len(set(loaded["training"]["track_ids"]) & set(loaded["validation"]["track_ids"])) == 0
assert len(set(loaded["training"]["track_ids"]) & set(loaded["test"]["track_ids"])) == 0
assert len(set(loaded["validation"]["track_ids"]) & set(loaded["test"]["track_ids"])) == 0
for s in loaded:
    assert loaded[s]["X"].shape[0] == len(loaded[s]["track_ids"])
    assert loaded[s]["X"].shape[1] == len(keep_columns)
    assert np.isfinite(loaded[s]["X"]).all()
print("All checks passed!")
print("Train feature mean (first 5):", np.round(loaded["training"]["X"].mean(axis=0)[:5], 3))
print("Train feature std  (first 5):", np.round(loaded["training"]["X"].std(axis=0)[:5], 3))

## 8. Optional quick visualization

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist(loaded["training"]["X"][:, 0], bins=50)
plt.title("Distribution of first standardized audio feature (training)")
plt.xlabel("Standardized feature value")
plt.ylabel("Number of tracks")
plt.tight_layout()
plt.show()

## 9. Handoff to the autoencoder teammate

Your teammate can use the following code in `04_autoencoder.ipynb` (after finding the repository root as in Section 1):

In [ ]:
import numpy as np
import torch

train = np.load(OUTPUT_DIR / "training.npz")
val = np.load(OUTPUT_DIR / "validation.npz")
test = np.load(OUTPUT_DIR / "test.npz")

X_train = torch.from_numpy(train["X"])
X_val = torch.from_numpy(val["X"])
X_test = torch.from_numpy(test["X"])
input_dim = X_train.shape[1]

print("Autoencoder input dimension:", input_dim)
print("Training:", tuple(X_train.shape))
print("Validation:", tuple(X_val.shape))
print("Test:", tuple(X_test.shape))

### Notes for collaboration

- Do **not** commit downloaded FMA data, processed `.npz` files, or fitted preprocessing artifacts if they are gitignored. Each teammate should run the notebook locally/Colab or obtain processed files through an agreed shared location.
- Keep the official split definitions fixed when comparing the baseline and autoencoder.
- Preprocessing uses no genre or tag labels; these remain available separately for evaluation.
- Standardized features are not bounded to `[0, 1]`; for the autoencoder, a **linear output layer** with an MSE reconstruction loss is a reasonable starting point.
- This pipeline handles tabular precomputed features, **not raw MP3 waveforms**.
